# Camada Preventiva de Priorização Territorial (Conecta Recife)
## Sprints 1 a 3: preparação dos dados e agrupamento territorial

Este notebook roda no **Google Colab** e também localmente (VS Code). Ele executa as 11 etapas de limpeza do plano, registra as perdas em cada etapa e salva as bases tratadas. A sprint 3 agrupa as denúncias georreferenciadas em pontos com histórico básico.

**Como usar:** rode as células em ordem (*Ambiente de execução → Executar tudo*). A etapa 4 classifica as situações conhecidas e interrompe a execução se houver casos novos, inválidos ou conflitantes; o CSV gerado permite revisar a classificação. Há também uma revisão manual na etapa 10 (decodificar `TIPO` da drenagem).

**Arquivos esperados em `dados/bruto/`:** `156_2023.csv`, `156_2024.csv`, `156_2025.csv`, `bairros.geojson`, `drenagem_elementos.geojson`, `drenagem_rede.geojson`, `setores_coleta.geojson`, `roteiros_coleta.csv`, `onde_descartar.csv` e, opcionalmente, `defesa_civil_2023.csv`, `defesa_civil_2024.csv`, `defesa_civil_2025.csv`.

**Recorte piloto:** Drenagem na RPA 4, entre 2023 e 2025. A categoria é selecionada pelo nome exato, sem incluir Macrodrenagem. A RPA é filtrada antes das demais etapas de limpeza e das estatísticas. Para alterar o piloto, edite `GRUPOS_RECORTE` e `RPA_PILOTO` na célula de parâmetros, mantendo uma categoria e uma RPA.


# Sprint 1: ambiente, caminhos e conferência dos dados

In [ ]:
# ===== AMBIENTE E CAMINHOS =====
%pip install -q geopandas pyarrow

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    EM_COLAB = True
    # ajuste se a sua pasta no Drive tiver outro nome
    PASTA = '/content/drive/MyDrive/projetos5-scope/camada_preventiva'
except ImportError:
    EM_COLAB = False
    # fora do Colab (VS Code): sobe a partir da pasta atual até achar a pasta que contém 'dados'
    # (assim funciona mesmo com o notebook dentro de notebooks/). Ou use a variável PASTA_LOCAL.
    PASTA = os.environ.get('PASTA_LOCAL')
    if not PASTA:
        PASTA = os.path.abspath(os.getcwd())
        while True:
            if os.path.isdir(os.path.join(PASTA, 'dados')):
                break
            interna = os.path.join(PASTA, 'camada_preventiva')
            if os.path.isdir(os.path.join(interna, 'dados')):
                PASTA = interna
                break
            pai = os.path.dirname(PASTA)
            if pai == PASTA:
                break
            PASTA = pai

# Opcional: aponte para outra pasta de dados brutos (ex.: Google Drive para computador). None = padrão.
BRUTO_CUSTOM = None   # ex.: r'G:\Meu Drive\NOME_DA_PASTA\dados\bruto'

BRUTO  = BRUTO_CUSTOM or f'{PASTA}/dados/bruto'
PROC   = f'{PASTA}/dados/processado'
SAIDAS = f'{PASTA}/saidas'
# NÃO cria dados/bruto sozinho: se ela não existir, avisa o que há na pasta e o que fazer
if not os.path.isdir(BRUTO):
    dados = os.path.join(PASTA, 'dados')
    msg = f'Não achei a pasta {BRUTO}\n'
    msg += f'Conteúdo de {PASTA}: {sorted(os.listdir(PASTA)) if os.path.isdir(PASTA) else "(não existe)"}\n'
    if os.path.isdir(dados):
        msg += f'Conteúdo de {dados}: {sorted(os.listdir(dados))}\n'
    if EM_COLAB:
        msg += ('Colab: confira PASTA. Se o projeto está em "Compartilhados comigo", adicione um atalho dele '
                'ao Meu Drive (com outro nome) e use o caminho do atalho.')
    else:
        msg += ('VS Code: crie a pasta dados/bruto dentro de camada_preventiva e coloque nela os arquivos brutos '
                '(156_2023.csv, 156_2024.csv, 156_2025.csv, bairros.geojson, drenagem_elementos.geojson, ...).')
    raise FileNotFoundError(msg)
for p in (PROC, SAIDAS):
    os.makedirs(p, exist_ok=True)

# ===== PARÂMETROS DO PROJETO (calibre aqui) =====
ANOS = [2023, 2024, 2025]
GRUPOS_RECORTE = ['DRENAGEM']                     # uma categoria; nome exato da tabela de grupos
RPA_PILOTO = 4                                    # uma RPA piloto (1 a 6)
RAIO_M = 80                                      # sprint 3: raio fixo em metros
AREA_MIN_M2 = 500                                 # etapa 9: partes de setor menores que isso são removidas
CRS_M = 'EPSG:31985'                              # SIRGAS 2000 / UTM 25S (metros)

print('Pasta do projeto:', PASTA)
arquivos = sorted(os.listdir(BRUTO))
print(arquivos)
if not arquivos:
    print('\nATENÇÃO: a pasta dados/bruto está VAZIA. Os arquivos não estão neste caminho.')
    print('Se a pasta do projeto está em "Compartilhados comigo", adicione um atalho dela ao Meu Drive')
    print('e ajuste PASTA. Depois rode: drive.mount("/content/drive", force_remount=True)')

In [ ]:
# ===== FUNÇÕES AUXILIARES =====
import re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display

def sem_acento(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))

def norm_texto(s):
    # MAIÚSCULO, sem acento, sem espaços repetidos
    if pd.isna(s):
        return s
    return re.sub(r'\s+', ' ', sem_acento(s).upper().strip())

def norm_coluna(c):
    return re.sub(r'\W+', '_', norm_texto(c)).strip('_')

ABREV = {r'\bAV\b': 'AVENIDA', r'\bR\b': 'RUA', r'\bTV\b': 'TRAVESSA', r'\bPC\b': 'PRACA',
         r'\bEST\b': 'ESTRADA', r'\bAL\b': 'ALAMEDA', r'\bLG\b': 'LARGO', r'\bDR\b': 'DOUTOR',
         r'\bPROF\b': 'PROFESSOR', r'\bCEL\b': 'CORONEL', r'\bSTA\b': 'SANTA', r'\bSTO\b': 'SANTO',
         r'\b1TR\b': 'PRIMEIRA TRAVESSA', r'\b2TR\b': 'SEGUNDA TRAVESSA', r'\b3TR\b': 'TERCEIRA TRAVESSA'}

def expandir_abrev(s):
    if pd.isna(s):
        return s
    for pat, rep in ABREV.items():
        s = re.sub(pat, rep, s)
    return s

def ler_csv(caminho):
    # tenta combinações de encoding e separador e diz o motivo real se falhar
    if not os.path.exists(caminho):
        pasta = os.path.dirname(caminho)
        existe = sorted(os.listdir(pasta)) if os.path.isdir(pasta) else 'a pasta nem existe'
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho}\nConteúdo da pasta: {existe}')
    ultimo = None
    for enc in ('utf-8', 'utf-8-sig', 'latin-1'):
        for sep in (';', ','):
            try:
                d = pd.read_csv(caminho, sep=sep, encoding=enc, dtype=str, low_memory=False)
                if d.shape[1] > 1:
                    return d
            except Exception as e:
                ultimo = e
    raise ValueError(f'O arquivo existe, mas não consegui ler: {caminho}\nÚltimo erro: {ultimo}')

def parse_data(s):
    s = s.astype('string').str.strip()
    iso = s.str.match(r'^\d{4}-\d{2}-\d{2}', na=False)
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    out[iso] = pd.to_datetime(s[iso].str[:10], format='%Y-%m-%d', errors='coerce')
    out[~iso] = pd.to_datetime(s[~iso].str[:10], format='%d/%m/%Y', errors='coerce')
    return out

def achar_coluna(df, *candidatas):
    for c in candidatas:
        if c in df.columns:
            return c
    raise KeyError(f'Nenhuma destas colunas existe: {candidatas}. Colunas disponíveis: {list(df.columns)}')

def para_numero(s):
    return pd.to_numeric(s.astype('string').str.replace(',', '.', regex=False), errors='coerce')

# tabela de perdas: registra quantas linhas entram e saem em cada etapa
perdas = []
def log(etapa, antes, depois, obs=''):
    perdas.append({'etapa': etapa, 'entrou': antes, 'saiu': depois, 'perdeu': antes - depois, 'obs': obs})
    print(f'[{etapa}] {antes:,} -> {depois:,} (perdeu {antes - depois:,}) {obs}')

In [ ]:
# ===== CONFERÊNCIA DOS ARQUIVOS BRUTOS =====
for nome in sorted(os.listdir(BRUTO)):
    caminho = f'{BRUTO}/{nome}'
    mb = os.path.getsize(caminho) / 1e6
    try:
        if nome.endswith('.geojson'):
            g = gpd.read_file(caminho)
            print(f'OK   {nome:34} {mb:7.1f} MB  {len(g):>9,} feições  CRS={g.crs}')
        elif nome.endswith('.csv'):
            d = ler_csv(caminho)
            print(f'OK   {nome:34} {mb:7.1f} MB  {len(d):>9,} linhas  {d.shape[1]} colunas')
        else:
            print(f'??   {nome:34} {mb:7.1f} MB  extensão não reconhecida')
    except Exception as e:
        print(f'ERRO {nome:34} {mb:7.1f} MB  {e}')

# Sprint 2: limpeza e normalização do 156 (etapas 1 a 7)

In [ ]:
# ===== ETAPA 0: juntar 2023, 2024 e 2025 =====
partes = []
for ano in ANOS:
    d = ler_csv(f'{BRUTO}/156_{ano}.csv')
    d.columns = [norm_coluna(c) for c in d.columns]
    d['ANO_ARQUIVO'] = ano
    print(ano, d.shape)
    partes.append(d)

comuns = set.intersection(*[set(p.columns) for p in partes])
todas = set.union(*[set(p.columns) for p in partes])
if todas - comuns:
    print('ATENÇÃO: colunas que não existem em todos os anos:', sorted(todas - comuns))

df = pd.concat(partes, ignore_index=True)
log('0 juntar 2023-2025', len(df), len(df))
print(df.columns.tolist())

# grupos de serviço existentes: confira o nome exato em GRUPOS_RECORTE
df['_GRUPO'] = df[achar_coluna(df, 'GRUPOSERVICO_DESCRICAO')].map(norm_texto)
grupos = df['_GRUPO'].value_counts().rename_axis('grupo').reset_index(name='n')
grupos.to_csv(f'{SAIDAS}/grupos_servico.csv', index=False)
display(grupos.head(20))

In [ ]:
# ===== ETAPAS 1 e 2: filtrar o recorte e remover duplicatas =====
# O piloto usa uma categoria e uma RPA; evita processar a cidade toda por engano.
if len(GRUPOS_RECORTE) != 1:
    raise ValueError('Defina uma única categoria em GRUPOS_RECORTE.')
if RPA_PILOTO not in range(1, 7):
    raise ValueError('Defina RPA_PILOTO com um número de 1 a 6.')

grupos_recorte = [norm_texto(g) for g in GRUPOS_RECORTE]
n = len(df)
# Nome exato: DRENAGEM não inclui MACRODRENAGEM.
df = df[df['_GRUPO'].isin(grupos_recorte)].copy()
log('1 filtrar categoria', n, len(df), f'grupos={grupos_recorte}')
if df.empty:
    raise ValueError('O filtro zerou a base: ajuste GRUPOS_RECORTE com base na tabela de grupos acima.')
df['GRUPOSERVICO_DESCRICAO'] = df['_GRUPO']

df['RPA_NUM'] = pd.to_numeric(df[achar_coluna(df, 'RPA')].astype('string').str.extract(r'(\d+)')[0], errors='coerce')
n = len(df)
df = df[df['RPA_NUM'] == RPA_PILOTO].copy()
log(f'1b recorte RPA {RPA_PILOTO}', n, len(df))
if df.empty:
    raise ValueError('Não há registros da categoria na RPA escolhida. Confira o recorte.')

n = len(df)
df = df.drop_duplicates()
log('2 duplicatas exatas', n, len(df))


In [ ]:
# ===== ETAPA 3: converter datas e descartar as fora do ano do arquivo =====
df['DATA_DEMANDA'] = parse_data(df[achar_coluna(df, 'DATA_DEMANDA')])
df['DATA_ULT_SITUACAO'] = parse_data(df[achar_coluna(df, 'DATA_ULT_SITUACAO', 'DATA_ULTIMA_SITUACAO')])

n = len(df)
df = df[df['DATA_DEMANDA'].dt.year == df['ANO_ARQUIVO']].copy()
log('3 datas inválidas ou fora do ano', n, len(df))

In [ ]:
# ===== ETAPA 4: classificar SITUACAO em FINAL ou ABERTA =====
# Guarda o valor recebido antes de corrigir as variações de escrita.
if 'SITUACAO_ORIGINAL' not in df.columns:
    df['SITUACAO_ORIGINAL'] = df[achar_coluna(df, 'SITUACAO')]
variacoes_sit = {'PREPARAO': 'PREPARACAO', 'EXECUO': 'EXECUCAO', 'FISCALIZAO': 'FISCALIZACAO'}
df['SITUACAO'] = df[achar_coluna(df, 'SITUACAO')].map(norm_texto).replace(variacoes_sit)
# Limpa uma classificação anterior caso esta célula pare por algum erro.
df['SITUACAO_TIPO'] = pd.NA

# Regra do projeto: atendida é final; as demais situações conhecidas estão abertas.
mapa_padrao = {'ATENDIDA': 'FINAL', 'CADASTRADA': 'ABERTA', 'PREPARACAO': 'ABERTA',
               'EXECUCAO': 'ABERTA', 'FISCALIZACAO': 'ABERTA',
               'PENDENCIA': 'ABERTA', 'PENDENTE': 'ABERTA'}
arq_sit = f'{SAIDAS}/situacoes_unicas.csv'
sit = df['SITUACAO'].value_counts(dropna=False).rename_axis('situacao').reset_index(name='n')
if os.path.exists(arq_sit):
    # Preserva decisões manuais e normaliza também os nomes já salvos no CSV.
    antigo = pd.read_csv(arq_sit, dtype=str, keep_default_na=False)[['situacao', 'tipo']]
    antigo['situacao'] = antigo['situacao'].map(norm_texto).replace(variacoes_sit)
    antigo['tipo'] = antigo['tipo'].map(norm_texto)
    antigo = antigo[antigo['tipo'] != ''].copy()
    conflitos = antigo.groupby('situacao')['tipo'].nunique()
    if (conflitos > 1).any():
        raise ValueError(f'Há classificações conflitantes em {arq_sit}. Corrija o CSV e rode esta célula novamente.')
    antigo = antigo.drop_duplicates('situacao')
    sit = sit.merge(antigo, on='situacao', how='left')
else:
    sit['tipo'] = pd.NA
sit['tipo'] = sit['tipo'].fillna(sit['situacao'].map(mapa_padrao)).fillna('')
sit.to_csv(arq_sit, index=False)
display(sit)

mapa_sit = dict(zip(sit['situacao'], sit['tipo']))
df['SITUACAO_TIPO'] = df['SITUACAO'].map(mapa_sit)
validas = df['SITUACAO_TIPO'].isin(['FINAL', 'ABERTA']) & df['SITUACAO'].fillna('').ne('')
df.loc[~validas, 'SITUACAO_TIPO'] = pd.NA
pendentes = int((~validas).sum())
log('4 classificar SITUACAO', len(df), len(df), f'{pendentes:,} linhas sem classificação válida (FINAL/ABERTA)')
if pendentes:
    raise ValueError(f'PAUSA MANUAL: há {pendentes:,} linhas sem situação ou classificação válida. '
                     f'Confira a origem e preencha "tipo" (FINAL ou ABERTA) em {arq_sit}; '
                     'depois rode esta célula novamente.')
print('Todas as situações estão classificadas.')


In [ ]:
# ===== ETAPAS 5 e 6: validar coordenadas e tratar as faltantes =====
df['LAT'] = para_numero(df[achar_coluna(df, 'LATITUDE', 'LAT')])
df['LON'] = para_numero(df[achar_coluna(df, 'LONGITUDE', 'LON', 'LONG')])
sem_coord = df[['LAT', 'LON']].isna().any(axis=1)

# limite real do Recife = união dos bairros
bairros = gpd.read_file(f'{BRUTO}/bairros.geojson')
if bairros.crs is None:
    bairros = bairros.set_crs(4326)
bairros = bairros.to_crs(4326)
limite = bairros.union_all() if hasattr(bairros, 'union_all') else bairros.unary_union

ok = ~sem_coord
pts = gpd.GeoSeries(gpd.points_from_xy(df.loc[ok, 'LON'], df.loc[ok, 'LAT']), crs=4326, index=df.index[ok])
dentro = pts.within(limite)

df['COORD_VALIDA'] = False
df.loc[dentro[dentro].index, 'COORD_VALIDA'] = True

# etapa 5: ponto fora do Recife não entra na análise por ponto (a linha é mantida e marcada)
log('5 validar coordenadas (limite do Recife)', int(ok.sum()), int(df['COORD_VALIDA'].sum()),
    'fora do limite: linhas mantidas, marcadas COORD_VALIDA=False')
# etapa 6: sem coordenada fica fora da análise por ponto, mas continua nas contagens por bairro
log('6 coordenadas faltantes', len(df), int(ok.sum()),
    'sem coordenada: mantidas para contagens por bairro, excluídas da análise por ponto')
print(f'Com coordenada válida: {df["COORD_VALIDA"].mean() * 100:.1f}% das linhas')

In [ ]:
# ===== ETAPA 7: padronizar texto (BAIRRO, LOGRADOURO, NUMERO) =====
for c in ('BAIRRO', 'LOGRADOURO'):
    if c in df.columns:
        df[c] = df[c].map(norm_texto).map(expandir_abrev)
if 'NUMERO' in df.columns:
    df['NUMERO'] = df['NUMERO'].map(norm_texto).replace({'SN': pd.NA, 'S/N': pd.NA})

# RPA_NUM já foi criada e filtrada na etapa 1.

df = df.drop(columns=['_GRUPO']).reset_index(drop=True)
df.insert(0, 'ID_LINHA', np.arange(len(df)))
log('7 padronizar texto', len(df), len(df), 'BAIRRO, LOGRADOURO e NUMERO')
df.head()

# Sprint 2: bases de apoio (etapas 8 a 11)

In [ ]:
# ===== ETAPA 8: projetar tudo para SIRGAS 2000 / UTM 25S (EPSG:31985) =====
v = df[df['COORD_VALIDA']]
g = gpd.GeoDataFrame(v[[]], geometry=gpd.points_from_xy(v['LON'], v['LAT']), crs=4326).to_crs(CRS_M)
df['X'] = g.geometry.x
df['Y'] = g.geometry.y

def abrir(nome):
    x = gpd.read_file(f'{BRUTO}/{nome}')
    if x.crs is None:
        x = x.set_crs(4326)
    x.columns = [c if c == 'geometry' else norm_coluna(c) for c in x.columns]
    return x.to_crs(CRS_M)

elementos = abrir('drenagem_elementos.geojson')
rede = abrir('drenagem_rede.geojson')
setores = abrir('setores_coleta.geojson')
bairros_m = bairros.to_crs(CRS_M)

log('8 projetar para EPSG:31985', len(df), len(df),
    f'denúncias, {len(elementos):,} elementos, {len(rede):,} trechos, {len(setores):,} setores, {len(bairros_m):,} bairros')
print('Colunas dos setores:', setores.columns.tolist())
print('Colunas dos elementos:', elementos.columns.tolist())
print('Colunas da rede:', rede.columns.tolist())

In [ ]:
# ===== ETAPA 9: simplificar geometrias dos setores de coleta =====
# remove partes de polígono com área irrisória, para um setor não "aparecer" em lugar errado
candidatas = ['SETOR', 'SETOR_COLETA', 'COD_SETOR', 'ID_SETOR', 'CODIGO', 'NOME']
COL_SETOR = next((c for c in candidatas if c in setores.columns), None)
print('Coluna do setor usada:', COL_SETOR, '(se for None, ajuste a lista acima com o nome real)')

exploded = setores.explode(index_parts=False)
exploded['AREA_M2'] = exploded.area
n = len(exploded)
exploded = exploded[exploded['AREA_M2'] >= AREA_MIN_M2]
if COL_SETOR:
    setores = exploded.dissolve(by=COL_SETOR, as_index=False)
else:
    setores = exploded.reset_index(drop=True)
log('9 simplificar setores de coleta', n, len(exploded), f'partes < {AREA_MIN_M2} m² removidas')

In [ ]:
# ===== ETAPA 10 (parte 1): listar os códigos de TIPO da drenagem =====
COL_TIPO = 'TIPO' if 'TIPO' in elementos.columns else None
if COL_TIPO is None:
    raise KeyError(f'Coluna TIPO não encontrada. Colunas: {elementos.columns.tolist()}')

arq_tipos = f'{SAIDAS}/tipos_drenagem.csv'
tipos = elementos[COL_TIPO].astype(str).value_counts().rename_axis('TIPO').reset_index(name='n')
if os.path.exists(arq_tipos):
    antigo = pd.read_csv(arq_tipos, dtype=str)[['TIPO', 'significado', 'conta_exposicao']]
    tipos = tipos.merge(antigo, on='TIPO', how='left')
else:
    tipos['significado'] = ''
    tipos['conta_exposicao'] = ''
tipos = tipos.fillna('')
tipos.to_csv(arq_tipos, index=False)
display(tipos)
print(f'PAUSA MANUAL: com a equipe da EMLURB ou a documentação, preencha "significado" e '
      f'"conta_exposicao" (SIM ou NAO) em {arq_tipos}. Só contam os elementos que recebem água da rua. '
      f'Depois rode a próxima célula.')

In [ ]:
# ===== ETAPA 10 (parte 2): manter só os elementos que recebem água da rua =====
tipos = pd.read_csv(arq_tipos, dtype=str).fillna('')
validos = tipos.loc[tipos['conta_exposicao'].str.upper().str.strip().isin(['SIM', 'S']), 'TIPO']

n = len(elementos)
if len(validos) == 0:
    print('AVISO: "conta_exposicao" ainda está vazia. Nenhum filtro foi aplicado.')
    log('10 decodificar drenagem (TIPO)', n, n, 'pendente: preencher tipos_drenagem.csv')
else:
    elementos = elementos[elementos[COL_TIPO].astype(str).isin(validos)].copy()
    log('10 decodificar drenagem (só elementos que recebem água)', n, len(elementos),
        f'TIPOS mantidos: {sorted(validos)}')

In [ ]:
# ===== ETAPA 11: filtrar o descarte (só pontos que aceitam volumosos e entulho) =====
descarte = ler_csv(f'{BRUTO}/onde_descartar.csv')
descarte.columns = [norm_coluna(c) for c in descarte.columns]
print('Colunas:', descarte.columns.tolist())

COL_RES = next((c for c in descarte.columns if 'RESIDUO' in c), None)
if COL_RES is None:
    raise KeyError(f'Não achei a coluna de tipo de resíduo. Colunas: {descarte.columns.tolist()}')
display(descarte[COL_RES].value_counts().head(20))

n = len(descarte)
txt = descarte[COL_RES].map(norm_texto).fillna('')
descarte = descarte[txt.str.contains('VOLUMOSO|ENTULHO')].copy().reset_index(drop=True)
log('11 filtrar descarte (volumosos e entulho)', n, len(descarte))
if descarte.empty:
    print('AVISO: nenhum ponto aceitou volumosos/entulho. Confira os valores de tipo de resíduo acima.')

# coordenadas em metros
descarte['LAT'] = para_numero(descarte[achar_coluna(descarte, 'LATITUDE', 'LAT')])
descarte['LON'] = para_numero(descarte[achar_coluna(descarte, 'LONGITUDE', 'LON', 'LONG')])
gd = gpd.GeoDataFrame(descarte[[]], geometry=gpd.points_from_xy(descarte['LON'], descarte['LAT']),
                      crs=4326).to_crs(CRS_M)
descarte['X'] = gd.geometry.x
descarte['Y'] = gd.geometry.y
descarte.head()

In [ ]:
# ===== (OPCIONAL) DEFESA CIVIL: leitura e padronização para a checagem por bairro =====
dc_partes = []
for ano in ANOS:
    arq = f'{BRUTO}/defesa_civil_{ano}.csv'
    if not os.path.exists(arq):
        print(f'{ano}: arquivo ausente, pulando')
        continue
    d = ler_csv(arq)
    d.columns = [norm_coluna(c) for c in d.columns]
    d['ANO_ARQUIVO'] = ano
    print(ano, d.shape)
    dc_partes.append(d)

if dc_partes:
    dc = pd.concat(dc_partes, ignore_index=True)
    dc['DATA'] = parse_data(dc[achar_coluna(dc, 'DATA', 'DATA_DA_ACAO')])
    n = len(dc)
    dc = dc[dc['DATA'].dt.year == dc['ANO_ARQUIVO']].copy()   # a coluna Ano do portal é inconsistente: vale a data
    log('7 defesa civil: datas dentro do ano', n, len(dc))

    limpa = lambda s: re.sub(r'[^A-Z0-9 ,./-]', '', s) if isinstance(s, str) else s   # tira lixo de codificação
    for c in ('BAIRRO', 'ENDERECO'):
        if c in dc.columns:
            dc[c] = dc[c].map(norm_texto).map(limpa).map(expandir_abrev)
    dc.to_parquet(f'{PROC}/defesa_civil_limpo.parquet', index=False)
    print(dc.groupby('ANO_ARQUIVO').size())
else:
    print('Sem arquivos da Defesa Civil: etapa pulada (não bloqueia as demais).')

# Estatísticas descritivas e mapa

In [ ]:
# ===== ESTATÍSTICAS DESCRITIVAS =====
df['MES'] = df['DATA_DEMANDA'].dt.to_period('M').astype(str)
mensal = df.groupby('MES').size()
mensal.to_csv(f'{SAIDAS}/estat_volume_mensal.csv')
df.groupby([df['DATA_DEMANDA'].dt.year, df['DATA_DEMANDA'].dt.month]).size().unstack(0) \
  .to_csv(f'{SAIDAS}/estat_mes_x_ano.csv')
if 'BAIRRO' in df.columns:
    df['BAIRRO'].value_counts().head(20).to_csv(f'{SAIDAS}/estat_top_bairros.csv')
if 'SERVICO_DESCRICAO' in df.columns:
    df['SERVICO_DESCRICAO'].value_counts().head(20).to_csv(f'{SAIDAS}/estat_top_servicos.csv')

print(f'Volume total: {len(df):,} | com coordenada válida: {df["COORD_VALIDA"].mean() * 100:.1f}%')
print('\nVolume por ano:\n', df.groupby('ANO_ARQUIVO').size())
print('\nTop 10 bairros:\n', df['BAIRRO'].value_counts().head(10))

fig, ax = plt.subplots(figsize=(10, 4))
mensal.plot(ax=ax, color='#888')
ax.set_title(f'Volume mensal de denúncias - {GRUPOS_RECORTE[0]} / RPA {RPA_PILOTO}')
ax.set_xlabel('')
ax.set_ylabel('denúncias')
plt.xticks(rotation=60)
plt.tight_layout()
fig.savefig(f'{SAIDAS}/volume_mensal.png', dpi=120)
plt.show()

In [ ]:
# ===== MAPA DE CONCENTRAÇÃO (hexágonos sobre o contorno dos bairros) =====
vv = df[df['COORD_VALIDA']]
fig, ax = plt.subplots(figsize=(7, 9))
bairros.plot(ax=ax, color='#efefef', edgecolor='none', zorder=1)                  # fundo
hb = ax.hexbin(vv['LON'], vv['LAT'], gridsize=60, bins='log', cmap='YlOrRd',
               mincnt=1, alpha=0.9, zorder=2)                                      # hexágonos
bairros.plot(ax=ax, facecolor='none', edgecolor='#444444', linewidth=0.5, zorder=3)  # contorno POR CIMA
fig.colorbar(hb, ax=ax, shrink=0.7, label='Número de denúncias por hexágono (escala log)')
minx, miny, maxx, maxy = bairros.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title(f'Concentração de denúncias - {GRUPOS_RECORTE[0]} / RPA {RPA_PILOTO}')
fig.savefig(f'{SAIDAS}/mapa_densidade.png', dpi=150, bbox_inches='tight')
plt.show()

# Fechamento: salvar as bases tratadas e a tabela de perdas

In [ ]:
# ===== SALVAR =====
# A classificação deve estar completa mesmo se a etapa 4 tiver sido pulada.
if 'SITUACAO_TIPO' not in df.columns or not df['SITUACAO_TIPO'].isin(['FINAL', 'ABERTA']).all():
    raise ValueError('Não é possível salvar: execute a etapa 4 e classifique todas as situações em FINAL ou ABERTA.')

tabela_perdas = pd.DataFrame(perdas)
display(tabela_perdas)

df.to_parquet(f'{PROC}/156_limpo.parquet', index=False)
setores.to_parquet(f'{PROC}/setores_coleta.parquet')
elementos.to_parquet(f'{PROC}/drenagem_elementos.parquet')
rede.to_parquet(f'{PROC}/drenagem_rede.parquet')
descarte.to_csv(f'{PROC}/descarte_filtrado.csv', index=False)
tabela_perdas.to_csv(f'{SAIDAS}/tabela_perdas.csv', index=False)

print('Arquivos em dados/processado:', sorted(os.listdir(PROC)))
print('Arquivos em saidas:', sorted(os.listdir(SAIDAS)))

# Sprint 3: agrupamento territorial (denúncia → ponto)

Usamos um raio fixo de `RAIO_M` metros (padrão: 80 m), com as coordenadas `X` e `Y` em EPSG:31985. A entrada é `dados/processado/156_limpo.parquet`. Somente registros com `COORD_VALIDA == True` participam do agrupamento; os demais continuam na base do piloto.

As denúncias são percorridas por data da demanda e `ID_LINHA`. A primeira denúncia ainda não agrupada vira o centro fixo de um ponto. As denúncias ainda disponíveis a até 80 m desse centro entram no mesmo ponto. O centro não se desloca: o grupo não cresce por uma cadeia de vizinhos. Denúncias isoladas também formam pontos.

Esta regra é simples e depende da ordem adotada. Duas denúncias do mesmo grupo podem ficar a até 160 m entre si, mas cada uma estará a até 80 m do centro. IDs de pontos se referem à execução e podem mudar se a base ou o raio mudar.

A classificação de situações já feita é usada no histórico. A padronização textual adicional de bairros e serviços permanece adiada; o agrupamento usa as coordenadas, sem depender desses textos. A RPA é a informada no 156, e a validação territorial continua sendo o limite do Recife.


In [ ]:
# ===== SPRINT 3: preparar os registros para agrupamento =====
if len(GRUPOS_RECORTE) != 1 or RPA_PILOTO not in range(1, 7):
    raise ValueError('Defina uma categoria e uma RPA piloto de 1 a 6.')
if not np.isfinite(RAIO_M) or RAIO_M <= 0:
    raise ValueError('RAIO_M deve ser um número positivo em metros.')

crs_pontos = gpd.GeoSeries([], crs=CRS_M).crs
if not crs_pontos.is_projected or any(e.unit_conversion_factor != 1 for e in crs_pontos.axis_info[:2]):
    raise ValueError('Use um sistema de coordenadas projetado em metros, como EPSG:31985.')

base_pontos = pd.read_parquet(f'{PROC}/156_limpo.parquet')
if base_pontos['ID_LINHA'].isna().any() or base_pontos['ID_LINHA'].duplicated().any():
    raise ValueError('ID_LINHA deve identificar cada denúncia uma única vez. Refaça a limpeza.')
if (not base_pontos['RPA_NUM'].eq(RPA_PILOTO).fillna(False).all() or
        not base_pontos['GRUPOSERVICO_DESCRICAO'].isin([norm_texto(g) for g in GRUPOS_RECORTE]).all()):
    raise ValueError('A base salva não corresponde ao piloto. Execute novamente as etapas anteriores.')

# Os registros sem coordenada válida continuam na base original, sem ID_PONTO.
denuncias_pontos = base_pontos[base_pontos['COORD_VALIDA'].eq(True)].copy()
if denuncias_pontos.empty:
    raise ValueError('Não há denúncias com coordenadas válidas para agrupar.')
if not np.isfinite(denuncias_pontos[['X', 'Y']].to_numpy(dtype=float)).all():
    raise ValueError('Há coordenadas X/Y inválidas. Refaça a etapa de projeção.')
if not denuncias_pontos['SITUACAO_TIPO'].isin(['FINAL', 'ABERTA']).all():
    raise ValueError('Classifique todas as situações antes de gerar o histórico por ponto.')
if denuncias_pontos[['DATA_DEMANDA', 'DATA_ULT_SITUACAO']].isna().any().any():
    raise ValueError('Há datas ausentes. Confira a etapa de conversão das datas.')

denuncias_pontos = denuncias_pontos.sort_values(['DATA_DEMANDA', 'ID_LINHA']).reset_index(drop=True)
print(f'Denúncias do piloto: {len(base_pontos):,}')
print(f'Entram no agrupamento: {len(denuncias_pontos):,}')
print(f'Sem coordenada válida: {len(base_pontos) - len(denuncias_pontos):,} (mantidas na base original)')


In [ ]:
# ===== SPRINT 3: agrupar por distância a um centro fixo =====
denuncias_pontos['ID_PONTO'] = pd.Series(pd.NA, index=denuncias_pontos.index, dtype='Int64')
centros = []
for i, denuncia in denuncias_pontos.iterrows():
    if pd.notna(denuncias_pontos.at[i, 'ID_PONTO']):
        continue
    id_ponto = len(centros) + 1
    distancia2 = ((denuncias_pontos['X'] - denuncia['X']) ** 2 +
                  (denuncias_pontos['Y'] - denuncia['Y']) ** 2)
    vizinhas = denuncias_pontos['ID_PONTO'].isna() & (distancia2 <= RAIO_M ** 2)
    denuncias_pontos.loc[vizinhas, 'ID_PONTO'] = id_ponto
    centros.append({'ID_PONTO': id_ponto, 'ID_LINHA_REFERENCIA': denuncia['ID_LINHA'],
                    'X': denuncia['X'], 'Y': denuncia['Y'], 'BAIRRO': denuncia['BAIRRO'],
                    'RPA_NUM': denuncia['RPA_NUM'],
                    'GRUPOSERVICO_DESCRICAO': denuncia['GRUPOSERVICO_DESCRICAO']})

pontos = pd.DataFrame(centros)
# Distância ao centro permite conferir a regra dos 80 m em cada denúncia.
denuncias_pontos = denuncias_pontos.merge(
    pontos[['ID_PONTO', 'X', 'Y']].rename(columns={'X': 'X_CENTRO', 'Y': 'Y_CENTRO'}),
    on='ID_PONTO', how='left', validate='many_to_one')
denuncias_pontos['DISTANCIA_M'] = np.sqrt(
    (denuncias_pontos['X'] - denuncias_pontos['X_CENTRO']) ** 2 +
    (denuncias_pontos['Y'] - denuncias_pontos['Y_CENTRO']) ** 2)
print(f'Pontos agrupados: {len(pontos):,} | raio: {RAIO_M} m')


In [ ]:
# ===== SPRINT 3: histórico básico por ponto =====
resumo_pontos = denuncias_pontos.groupby('ID_PONTO').agg(
    N_DENUNCIAS=('ID_LINHA', 'size'),
    PRIMEIRA_DATA=('DATA_DEMANDA', 'min'),
    ULTIMA_DATA=('DATA_DEMANDA', 'max'),
    ULTIMA_ATUALIZACAO=('DATA_ULT_SITUACAO', 'max'),
    N_ABERTAS=('SITUACAO_TIPO', lambda s: int((s == 'ABERTA').sum())),
    N_FINAIS=('SITUACAO_TIPO', lambda s: int((s == 'FINAL').sum())),
    RAIO_MAX_M=('DISTANCIA_M', 'max')).reset_index()
pontos = pontos.merge(resumo_pontos, on='ID_PONTO', validate='one_to_one')

# Última situação observada: data de atualização, demanda e ID desempatam a escolha.
ultimas = denuncias_pontos.sort_values(['DATA_ULT_SITUACAO', 'DATA_DEMANDA', 'ID_LINHA'])     .drop_duplicates('ID_PONTO', keep='last')
ultimas = ultimas[['ID_PONTO', 'SITUACAO', 'SITUACAO_TIPO']].rename(
    columns={'SITUACAO': 'ULTIMA_SITUACAO', 'SITUACAO_TIPO': 'ULTIMA_SITUACAO_TIPO'})
pontos = pontos.merge(ultimas, on='ID_PONTO', validate='one_to_one')
# Resumo das solicitações: basta uma aberta para o ponto ter pendência.
pontos['SITUACAO_PONTO'] = np.where(pontos['N_ABERTAS'] > 0, 'ABERTA', 'FINAL')
pontos['RAIO_M'] = RAIO_M
pontos = gpd.GeoDataFrame(pontos, geometry=gpd.points_from_xy(pontos['X'], pontos['Y']), crs=CRS_M)
pontos_latlon = pontos.to_crs(4326)
pontos['LAT'] = pontos_latlon.geometry.y
pontos['LON'] = pontos_latlon.geometry.x

display(pontos[['ID_PONTO', 'N_DENUNCIAS', 'N_ABERTAS', 'N_FINAIS',
                'PRIMEIRA_DATA', 'ULTIMA_DATA', 'SITUACAO_PONTO']].head(10))


In [ ]:
# ===== SPRINT 3: conferir e salvar os resultados =====
if (denuncias_pontos['ID_PONTO'].isna().any() or
        denuncias_pontos['ID_LINHA'].duplicated().any() or
        len(denuncias_pontos) != int(base_pontos['COORD_VALIDA'].eq(True).sum())):
    raise ValueError('Cada denúncia com coordenada válida deve pertencer a um único ponto.')
if int(pontos['N_DENUNCIAS'].sum()) != len(denuncias_pontos):
    raise ValueError('As contagens dos pontos não correspondem às denúncias agrupadas.')
if not (pontos['N_ABERTAS'] + pontos['N_FINAIS']).eq(pontos['N_DENUNCIAS']).all():
    raise ValueError('As situações não correspondem ao histórico dos pontos.')
if (denuncias_pontos['DISTANCIA_M'] > RAIO_M + 1e-6).any():
    raise ValueError('Há denúncias além do raio definido para seu centro.')

pontos.to_parquet(f'{PROC}/pontos_agrupados.parquet', index=False)
denuncias_pontos.to_parquet(f'{PROC}/denuncias_pontos.parquet', index=False)
pontos.drop(columns='geometry').to_csv(f'{SAIDAS}/pontos_agrupados.csv', index=False)
print(f'Histórico preservado: {len(denuncias_pontos):,} denúncias em {len(pontos):,} pontos.')
print(f'Pontos com uma única denúncia: {int(pontos["N_DENUNCIAS"].eq(1).sum()):,}')
print('Arquivos gerados: pontos_agrupados.parquet, denuncias_pontos.parquet e pontos_agrupados.csv')


## Conferência visual de alguns pontos

A amostra inclui o ponto com mais denúncias, o de maior distância ao centro e um dos de menor contagem, sem repetir IDs. Os gráficos mostram o raio de 80 m, as denúncias do grupo e as de outros grupos próximos. As coordenadas são distâncias em metros em relação ao centro de cada ponto. A tabela da amostra permite conferir também as datas e as situações.


In [ ]:
# ===== SPRINT 3: validar visualmente uma amostra =====
amostra_pontos = pd.concat([pontos.nlargest(1, 'N_DENUNCIAS'),
                           pontos.nlargest(1, 'RAIO_MAX_M'),
                           pontos.nsmallest(1, 'N_DENUNCIAS')]).drop_duplicates('ID_PONTO')
fig, axes = plt.subplots(1, len(amostra_pontos), figsize=(5 * len(amostra_pontos), 5), squeeze=False)
for ax, (_, ponto) in zip(axes.ravel(), amostra_pontos.iterrows()):
    membros = denuncias_pontos[denuncias_pontos['ID_PONTO'] == ponto['ID_PONTO']]
    distancia2 = ((denuncias_pontos['X'] - ponto['X']) ** 2 +
                  (denuncias_pontos['Y'] - ponto['Y']) ** 2)
    outras = denuncias_pontos[(denuncias_pontos['ID_PONTO'] != ponto['ID_PONTO']) &
                              (distancia2 <= (RAIO_M * 1.25) ** 2)]
    ax.scatter(outras['X'] - ponto['X'], outras['Y'] - ponto['Y'],
               color='#bbbbbb', s=20, label='Outro grupo')
    ax.scatter(membros['X'] - ponto['X'], membros['Y'] - ponto['Y'],
               color='#2878b5', s=30, alpha=0.7, label='Denúncias do ponto')
    ax.add_patch(plt.Circle((0, 0), RAIO_M, fill=False, color='#444444', linestyle='--'))
    ax.scatter([0], [0], marker='+', s=150, color='#c43c39', label='Centro fixo')
    ax.set_xlim(-RAIO_M * 1.25, RAIO_M * 1.25)
    ax.set_ylim(-RAIO_M * 1.25, RAIO_M * 1.25)
    ax.set_aspect('equal')
    ax.set_xlabel('Distância leste-oeste (m)')
    ax.set_ylabel('Distância norte-sul (m)')
    ax.set_title(f'Ponto {int(ponto["ID_PONTO"])} | {int(ponto["N_DENUNCIAS"])} denúncias\n'
                 f'{int(ponto["N_ABERTAS"])} abertas / {int(ponto["N_FINAIS"])} finais')
    ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()
fig.savefig(f'{SAIDAS}/validacao_pontos.png', dpi=150, bbox_inches='tight')
plt.show()
amostra_pontos.drop(columns='geometry').to_csv(f'{SAIDAS}/pontos_validacao.csv', index=False)
display(amostra_pontos[['ID_PONTO', 'N_DENUNCIAS', 'RAIO_MAX_M',
                        'PRIMEIRA_DATA', 'ULTIMA_DATA', 'ULTIMA_SITUACAO', 'SITUACAO_PONTO']])
